In [9]:
!pip install rapidfuzz sqlalchemy psycopg2-binary

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for psycopg2: filename=psycopg2-2.9.12-cp312-cp312-macosx_10_13_universal2.whl size=244166 sha256=4826780137f1fbb9e6eca10007dd6cdc776e4412b0b2400ebb82010c141d2b99
  Stored in directory: /Users/tiarasabrina/Library/Caches/pip/wheels/04/74/ff/720d90509a7e28eeecdaf470a74ba12afb31f8857624440482
Successfully built psycopg2

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import requests
import json
import time
from tqdm import tqdm

In [3]:
gold_company = pd.read_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx", engine="openpyxl")
gold_company

,Company ID IDX,Company ID BUMN,Nama Perusahaan IDX,Nama Perusahaan BUMN,Nama Perusahaan AHU,Jaro IDX_BUMN,Jaro IDX_AHU,Jaro BUMN_AHU,Sektor,Sub Sektor,...,Kode Emiten (Ticker),Tanggal IPO (Listed),Papan Pencatatan,Alamat Lengkap,Telepon,Fax,Email,Website,NPWP,Pair Category
0,1.001,NaN,PT Adaro Andalan Indonesia Tbk,NaN,PT Adaro Andalan Indonesia,0.0,1.0,0.0,Energi,"Minyak, Gas & Batu Bara",...,AADI,2024-12-05,Utama,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...,(021) 2553 3065 | 02125533000,(021) 2553 3066,corsec@adaroindonesia.com,www.adaroindonesia.com,02.433.115.9-091.000,IDX + AHU
1,1.002,NaN,Adaro Australia Pty Ltd,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
2,1.003,NaN,Adaro Capital Limited,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
3,1.004,NaN,Adaro International (Singapore) Pte Ltd,NaN,NaN,0.0,0.0,0.0,Perdagangan batubara,Perdagangan batubara,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
4,1.005,NaN,Arindo Holdings (Mauritius) Ltd,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772560,NaN,NaN,NaN,NaN,PT Fatima Trading Companies,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,"Gedung Fancy Mampang, Jl. Mampang Prpt. Raya N...",0881024990317,NaN,NaN,NaN,NaN,AHU Only
772561,NaN,NaN,NaN,NaN,PT Simpul Eventworks Indonesia,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,Jl. Wolter Monginsidi No.107A,087784451067,NaN,NaN,NaN,NaN,AHU Only
772562,NaN,NaN,NaN,NaN,PT Sinar Fajar Nusantara Kilat,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,Jalan Ancol Barat VIII Nomor 1 Kontener Nomor ...,081387119253,NaN,NaN,NaN,NaN,AHU Only
772563,NaN,NaN,NaN,NaN,PT Stasiun Hangat Sejahtera,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,"Infiniti Office, Bellezza BSA, 1st Floor Unit ...",082364055502,NaN,NaN,NaN,NaN,AHU Only


In [4]:
gold_company_address = gold_company[["Alamat Lengkap"]]
gold_company_address

,Alamat Lengkap
0,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...
1,NaN
2,NaN
3,NaN
4,NaN
...,...
772560,"Gedung Fancy Mampang, Jl. Mampang Prpt. Raya N..."
772561,Jl. Wolter Monginsidi No.107A
772562,Jalan Ancol Barat VIII Nomor 1 Kontener Nomor ...
772563,"Infiniti Office, Bellezza BSA, 1st Floor Unit ..."


In [11]:
import pandas as pd
from sqlalchemy import create_engine
from rapidfuzz import fuzz, process

# ============================================
# 1. LOAD MASTER DATA WILAYAH DARI POSTGRES
# ============================================
engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")

wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()

# Pisahin per level berdasarkan panjang kode (sesuaikan lagi setelah lo cek format kode aslinya)
def get_level(kode):
    parts = kode.split(".")
    if len(parts) == 1:
        return "provinsi"
    elif len(parts) == 2:
        return "kabkota"
    elif len(parts) == 3:
        return "kecamatan"
    elif len(parts) == 4:
        return "desa"
    return "unknown"

wilayah["level"] = wilayah["kode"].apply(get_level)

df_prov = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)

print(f"Provinsi: {len(df_prov)}, Kab/Kota: {len(df_kabkota)}, Kecamatan: {len(df_kec)}, Desa: {len(df_desa)}")


# ============================================
# 2. LOAD DATA ALAMAT MENTAH
# ============================================
gold_company = pd.read_excel(
    "/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx",
    engine="openpyxl"
)
gold_company_address = gold_company[["Alamat Lengkap"]].copy()
gold_company_address["alamat_clean"] = (
    gold_company_address["Alamat Lengkap"]
    .astype(str)
    .str.upper()
    .str.strip()
)


# ============================================
# 3. FUZZY MATCH HIERARKIS (provinsi -> kab/kota -> kecamatan -> desa)
# ============================================
def match_hierarchical(alamat, threshold=75):
    result = {
        "provinsi": None, "score_prov": 0,
        "kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "score_kec": 0,
        "desa": None, "score_desa": 0,
    }

    # skip kalau alamat kosong/NaN
    if not isinstance(alamat, str) or alamat.strip() == "" or alamat.upper() == "NAN":
        return result

    # --- Tahap provinsi ---
    match_prov = process.extractOne(
        alamat, df_prov["nama"], scorer=fuzz.partial_ratio, score_cutoff=threshold
    )
    if not match_prov:
        return result

    prov_nama, score_prov, idx_prov = match_prov
    kode_prov = df_prov.loc[idx_prov, "kode"]
    result["provinsi"] = prov_nama
    result["score_prov"] = score_prov

    # --- Tahap kab/kota ---
    kandidat_kabkota = df_kabkota[df_kabkota["kode"].str.startswith(kode_prov + ".")]
    if kandidat_kabkota.empty:
        return result

    match_kabkota = process.extractOne(
        alamat, kandidat_kabkota["nama"], scorer=fuzz.partial_ratio, score_cutoff=threshold
    )
    if not match_kabkota:
        return result

    kabkota_nama, score_kabkota, idx_kabkota = match_kabkota
    kode_kabkota = kandidat_kabkota.loc[idx_kabkota, "kode"]
    result["kabkota"] = kabkota_nama
    result["score_kabkota"] = score_kabkota

    # --- Tahap kecamatan ---
    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_kabkota + ".")]
    if kandidat_kec.empty:
        return result

    match_kec = process.extractOne(
        alamat, kandidat_kec["nama"], scorer=fuzz.partial_ratio, score_cutoff=threshold
    )
    if not match_kec:
        return result

    kec_nama, score_kec, idx_kec = match_kec
    kode_kec = kandidat_kec.loc[idx_kec, "kode"]
    result["kecamatan"] = kec_nama
    result["score_kec"] = score_kec

    # --- Tahap desa ---
    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_kec + ".")]
    if kandidat_desa.empty:
        return result

    match_desa = process.extractOne(
        alamat, kandidat_desa["nama"], scorer=fuzz.partial_ratio, score_cutoff=threshold
    )
    if match_desa:
        desa_nama, score_desa, idx_desa = match_desa
        result["desa"] = desa_nama
        result["score_desa"] = score_desa

    return result

# ============================================
# 4. APPLY KE SEMUA ROW
# ============================================
from tqdm import tqdm
tqdm.pandas()

results = gold_company_address["alamat_clean"].progress_apply(match_hierarchical)
result_df = pd.DataFrame(results.tolist())

final = pd.concat([gold_company_address, result_df], axis=1)
final.head(20)

Provinsi: 38, Kab/Kota: 514, Kecamatan: 7285, Desa: 83762


100%|██████████| 772565/772565 [01:59<00:00, 6456.87it/s]


,Alamat Lengkap,alamat_clean,provinsi,score_prov,kabkota,score_kabkota,kecamatan,score_kec,desa,score_desa
0,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...,CYBER 2 TOWER LANTAI 26\nJL. H.R. RASUNA SAID ...,PAPUA SELATAN,81.818182,NaN,0.0,NaN,0.0,NaN,0.0
1,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
2,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
3,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
4,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
5,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
6,JL. GURAMI NO. 173,JL. GURAMI NO. 173,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
7,"MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID B...","MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID B...",RIAU,75.000000,NaN,0.0,NaN,0.0,NaN,0.0
8,"GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID...","GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID...",RIAU,75.000000,NaN,0.0,NaN,0.0,NaN,0.0
9,"GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLO...","GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLO...",RIAU,75.000000,NaN,0.0,NaN,0.0,NaN,0.0


In [18]:
#final to excel
final.to_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/v2_Master_Matching_20260701_1059 (1)_matched.xlsx", index=False)